# Training Tasks Notebook

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from semantic_kernel.functions import kernel_function
from typing import Annotated
import random

class PokemonPlugin:
    @kernel_function(description="Provides a random pokemon name")
    def get_random_pokemon(self) -> Annotated[str, "Returns a random pokemon name"]:
        pokemon = ["Pikachu", "Charizard", "Bulbasaur"]

        return random.choice(pokemon)


In [3]:
from os import getenv
# Endpoint (LLM source)
from openai import AsyncOpenAI
# Service
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

client = AsyncOpenAI(
    api_key= getenv("GITHUB_TOKEN"),
    base_url= getenv("GITHUB_ENDPOINT")
)

service = OpenAIChatCompletion(
    ai_model_id= getenv("GITHUB_MODEL_ID"),
    async_client= client
)

In [4]:
import chromadb
from chromadb.api.models.Collection import Collection

collection = chromadb.PersistentClient(path="./chroma_db").create_collection(
    name="pokemon_documents",
    metadata={"description": "pokemon_service"},
    get_or_create=True,
)

documents = [
    "Pikachu is an electric type pokemon",
    "Charizard is a fire and flying type pokemon",
    "Bulbasaur is a grass and poison type pokemon",
    "Squirtle is a water type pokemon"
]

collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"source": "training", "type": "explanation"} for _ in documents]
)

class rag_plugin:
    def __init__(self, collection: Collection):
        self.collection = collection
    
    @kernel_function(name="get_pokemon_type", description="Retrieves the type of a pokemon from the database")
    def get_pokemon_type(self, query: str):
        rag_results = self.collection.query(
            query_texts=[query],
            include=["documents", "metadatas"],
            n_results=2
        )

        text_rag_result = ""
        if rag_results and rag_results.get("documents") and rag_results.get("metadatas")[0]:
            documents_list = rag_results.get("documents")[0]
            metadata_list = rag_results.get("metadatas")[0]

            for document, metadata in zip(documents_list, metadata_list):
                text_rag_result += f"Document: {document}\nMetadata: {metadata}\n\n"
        else:
            text_rag_result = "No retrieval context found."
        
        print(text_rag_result)
        return text_rag_result


In [5]:
from semantic_kernel.agents import ChatCompletionAgent

agent = ChatCompletionAgent(
    service= service,
    name= "PokemonAgent",
    instructions= """
    You are a helpful pokemon pokedex agent that gives descriptions of pokemon, if the user does not specify a pokemon name, describe a random pokemon.
    Put all sentences on a new line.
    Use the provided context, if context does not answer the query, use other tools.
    """,
    plugins= [PokemonPlugin(), rag_plugin(collection)]
)

In [6]:
user_inputs = [
    "Describe a pokemon", 
    "Describe the pokemon Squritle",
    "What type is the pokemon Charizard?"
]

async def main():
    thread = None

    for prompt in user_inputs:
        print(f"\n\n--- User: {prompt}\n", end="", flush=True)
        first_response = True

        async for response in agent.invoke_stream(messages=prompt, thread=thread):
            thread = response.thread

            if first_response:
                first_response = False
                print(f"--- {response.name}: {response}\n", end="", flush=True)

            print(response, end="", flush=True)
    
    if thread:
        await thread.delete()

await main()



--- User: Describe a pokemon
--- PokemonAgent: 
Document: Pikachu is an electric type pokemon
Metadata: {'source': 'training', 'type': 'explanation'}

Document: Charizard is a fire and flying type pokemon
Metadata: {'type': 'explanation', 'source': 'training'}


Pikachu is an Electric-type Pokémon.

Pikachu is known for its small size and adorable appearance.

It has yellow fur with long ears and a lightning bolt-shaped tail.

Pikachu is famous for its ability to generate electricity and unleash powerful electric attacks.

This Pokémon is often seen as a mascot of the Pokémon franchise.

--- User: Describe the pokemon Squritle
--- PokemonAgent: 
Document: Squirtle is a water type pokemon
Metadata: {'type': 'explanation', 'source': 'training'}

Document: Bulbasaur is a grass and poison type pokemon
Metadata: {'type': 'explanation', 'source': 'training'}


Squirtle is a Water-type Pokémon.

It is a small, turtle-like creature with blue skin.

Squirtle has large, expressive eyes and a s